In [1]:
import pandas as pd

# set current file path as working directory
import os

os.chdir('/home/nma/GIT/auto_WRF/auto_sim/modules')


In [2]:
dft = pd.DataFrame()

dft['locations'] = ['Thrissur', 'Brevard','Chennai']

date_today = pd.to_datetime('today')

dft['date'] = date_today.date()

dft['hour'] = date_today.hour

dft['forecast (days)'] = 3

dft

,locations,date,hour,forecast (days)
0,Thrissur,2023-06-21,11,3
1,Brevard,2023-06-21,11,3
2,Chennai,2023-06-21,11,3


In [3]:
dft['files'] = 'files' + os.sep + dft['locations'].astype(str) + os.sep


dft

,locations,date,hour,forecast (days),files
0,Thrissur,2023-06-21,11,3,files/Thrissur/
1,Brevard,2023-06-21,11,3,files/Brevard/
2,Chennai,2023-06-21,11,3,files/Chennai/


In [4]:
dft.to_csv('files/sim_dft.csv', index=False)

## editing namelist

In [5]:
dft['wps_namelist'] = dft['files'].astype(str) +  'namelist.wps'
dft['wrf_namelist'] = dft['files'].astype(str) +  'namelist.input'

#### For WPS

In [25]:
dft

start_date = str(dft['date'][0]) + '_' + str(dft['hour'][0]) + ':00:00'

end_date = str(dft['date'][0] + pd.Timedelta(days=dft['forecast (days)'][0])) + '_' + str(dft['hour'][0]) + ':00:00'

interval_seconds = dft['forecast (days)'][0] * 24 * 60 * 60

start_date, end_date , interval_seconds



def get_wps_time(dft):

    date = dft['date']
    hour = dft['hour']
    forecast_days = dft['forecast (days)']
    start_date = str(date) + '_' + str(hour) + ':00:00'
    end_date = str(date + pd.Timedelta(days=forecast_days)) + '_' + str(hour) + ':00:00'
    interval_seconds = forecast_days * 24 * 60 * 60

    dft_wps = pd.Series()
    dft_wps['start_date'] = start_date
    dft_wps['end_date'] = end_date
    dft_wps['interval_seconds'] = interval_seconds

    return dft_wps

dft_ = dft.set_index('locations')

dddd = dft_.apply(get_wps_time, axis=1)

dddd


,start_date,end_date,interval_seconds
locations,,,
Thrissur,2023-06-21_11:00:00,2023-06-24_11:00:00,259200
Brevard,2023-06-21_11:00:00,2023-06-24_11:00:00,259200
Chennai,2023-06-21_11:00:00,2023-06-24_11:00:00,259200


#### For WRF

In [26]:
spacing = "\t\t\t     "
run_days = dft['forecast (days)'][0]

run_hours = run_days * 24

run_minutes = 0

run_seconds = 0

start_year = str(dft['date'][0].year)

start_month = str(dft['date'][0].month)

start_day = str(dft['date'][0].day)

start_hour = str(dft['hour'][0])

start_minute = '00'

start_second = '00'

end_year = str((dft['date'][0] + pd.Timedelta(days=dft['forecast (days)'][0])).year)

end_month = str((dft['date'][0] + pd.Timedelta(days=dft['forecast (days)'][0])).month)

end_day = str((dft['date'][0] + pd.Timedelta(days=dft['forecast (days)'][0])).day)

end_hour = str(dft['hour'][0])

end_minute = '00'

end_second = '00'

interval_seconds = str(interval_seconds)


def edit_wrf_time(dft):
    
    
    date = dft['date']
    hour = dft['hour']
    forecast_days = dft['forecast (days)']

    spacing = "\t\t\t     "
    run_days = dft['forecast (days)']
    start_year,end_year = str(date.year) , str((date + pd.Timedelta(days=forecast_days)).year)
    start_month,end_month = str(date.month) , str((date + pd.Timedelta(days=forecast_days)).month)
    start_day,end_day = str(date.day) , str((date + pd.Timedelta(days=forecast_days)).day)
    start_hour,end_hour = str(hour) , str(hour)
    start_minute,end_minute = '00' , '00'
    start_second,end_second = '00' , '00'
    interval_seconds = str(forecast_days * 24 * 60 * 60)

    dft_wrf = pd.Series()
    dft_wrf['run_days'] = run_days
    dft_wrf['run_hours'] = run_hours
    dft_wrf['run_minutes'] = run_minutes
    dft_wrf['run_seconds'] = run_seconds
    dft_wrf['start_year'] = start_year
    dft_wrf['start_month'] = start_month
    dft_wrf['start_day'] = start_day
    dft_wrf['start_hour'] = start_hour
    dft_wrf['start_minute'] = start_minute
    dft_wrf['start_second'] = start_second
    dft_wrf['end_year'] = end_year
    dft_wrf['end_month'] = end_month
    dft_wrf['end_day'] = end_day
    dft_wrf['end_hour'] = end_hour
    dft_wrf['end_minute'] = end_minute
    dft_wrf['end_second'] = end_second
    dft_wrf['interval_seconds'] = interval_seconds


    
    
    return dft_wrf

ddd = dft_.apply(edit_wrf_time, axis=1)


ddd

,run_days,run_hours,run_minutes,run_seconds,start_year,start_month,start_day,start_hour,start_minute,start_second,end_year,end_month,end_day,end_hour,end_minute,end_second,interval_seconds
locations,,,,,,,,,,,,,,,,,
Thrissur,3,72,0,0,2023,6,21,11,00,00,2023,6,24,11,00,00,259200
Brevard,3,72,0,0,2023,6,21,11,00,00,2023,6,24,11,00,00,259200
Chennai,3,72,0,0,2023,6,21,11,00,00,2023,6,24,11,00,00,259200


In [ ]:

import re

file_test = '/home/nma/GIT/auto_WRF/auto_sim/modules/files/Thrissur/bk/namelist.wps'

with open(file_test, "r") as sources:
    namelist_comps = sources.read()

# find the line with start_day and all the spaces after it

start_day = re.findall(r'^start_day.*', 'start_day = 1,1,1')
spacing = "\t\t\t     "



new_start = re.sub(r'start_day.*', 'start_day' + spacing +'= 2,2,2,', namelist_comps)
new_year = re.sub(r'start_year.*', 'start_year' + spacing +'= 2019,2019,2019,', new_start)

# replace start_day with new value

#new_start_day = int(start_day) + 1

#new_namelist_comps = re.sub(r'start_day\s*=', 'start_day = ' + '4,4,4', namelist_comps)

print(new_year)

In [ ]:
wps_time_dft = dft_.apply(get_wps_time, axis=1)
dft_ind = dft.set_index('locations')
os.chdir('/home/nma/GIT/auto_WRF/auto_sim/modules/')


for row in wps_time_dft.index:

    file_name = dft_ind['wps_namelist'][row]

    with open(file_name, "r") as sources:
        namelist_comps = sources.read()

    for col in wps_time_dft.columns:
        spacing = "\t\t\t     "
        new_value = wps_time_dft[col][row]

    
    print(new_value)

In [41]:
ls

files/  __init__.py  main.py  module_workspaces.ipynb  setup_sim.py
